# Eval: Plausibility Pretesting (chạy **local**)

Chỉ notebook **deploy** cần Colab. Notebook này + summary chạy trên máy.

Cell đầu — 4 input: **MODEL**, **TOKEN**, **BASE_URL**, **MODE**.

**MODE** = điều kiện ablation (so với baseline paper):

| MODE | Schema | Thinking | Examples |
|---|---|---|---|
| ORIG | Không | Không | Có (baseline paper-like) |
| S | Có | Không | Có |
| T | Không | Có | Có |
| ST | Có | Có | Có |
| ST-E | Có | Có | Không |

Đổi `MODE` rồi chạy lại = một thí nghiệm mới → `results/<model>/<MODE>/`.

- **Resume mặc định:** câu đã có trong `scores.jsonl` (đủ `n_samples`) sẽ **bỏ qua**; call file lẻ còn thiếu sẽ tiếp tục.
- Raw evidence + tokens trong `calls/` — **không tính USD** (summary + `pricing.yaml`)
- Protocol: `configs/experiment.yaml`

- OpenRouter: cell **fetch providers** (price + latency) → cell **config** `OPENROUTER_PROVIDERS`.
- Parallel: `MAX_CONCURRENCY` (hoặc `max_concurrency` trong `experiment.yaml`) — số request đồng thời.
- Samples: `N_SAMPLES` (hoặc `n_samples` trong `experiment.yaml`) — số lần gọi API / câu.
- Reasoning: `REASONING_EFFORT` (hoặc `reasoning_effort` trong `experiment.yaml`) — chỉ khi MODE có thinking.
- Temperature: `TEMPERATURE` — `None` = auto (open 0.3 / closed 1.5); set số để ghi đè.


In [1]:
# === INPUTS (bắt buộc) ===
MODEL = "google/gemini-3.6-flash"
TOKEN = ""  # để trống → lấy từ doan/.env (OPENROUTER_API_KEY)
BASE_URL = "https://openrouter.ai/api/v1"
# Local Gemma: BASE_URL = "http://127.0.0.1:8080/v1"; MODEL = "gemma-4-12b"; TOKEN = "sk-local"
MODE = "T"

# Số sample / câu (None = lấy từ experiment.yaml, mặc định 20)
N_SAMPLES = 10

# Số request API chạy song song (None = lấy từ experiment.yaml, mặc định 8)
# Tài khoản mới ~10 RPM → giữ 1
MAX_CONCURRENCY = 1

# Nghỉ tối thiểu giữa 2 request (giây). 10 RPM → ~6–7s. None = experiment.yaml
REQUEST_DELAY_SEC = 0.5

# Temperature (None = auto theo BASE_URL: open 0.3 / closed 1.5 trong experiment.yaml)
TEMPERATURE = 1.5

# Reasoning effort khi MODE có thinking (T/ST/ST-E). None = experiment.yaml.
# OpenRouter: "max" | "xhigh" | "high" | "medium" | "low" | "minimal" | "none"
# ORIG / S bỏ qua (luôn tắt thinking).
REASONING_EFFORT = "medium"

# OPENROUTER_PROVIDERS → set ở cell "config providers" (sau khi fetch)


In [2]:
try:
    import openai
except ImportError:
    %pip install -q pyyaml tqdm pandas
    # Local package → import plausibility_eval resolves in kernel
    %pip install -q -e ../..


In [3]:
import os, sys
from pathlib import Path

# Locate doan/ repo root (local)
HERE = Path.cwd().resolve()
REPO = None
for c in [HERE, *HERE.parents]:
    if (c / "configs" / "experiment.yaml").exists():
        REPO = c
        break
    if (c / "doan" / "configs" / "experiment.yaml").exists():
        REPO = c / "doan"
        break
assert REPO is not None and (REPO / "configs" / "experiment.yaml").exists(), "Chạy notebook từ trong repo doan/"
sys.path.insert(0, str(REPO / "src"))
print("REPO:", REPO)

# Load doan/.env into os.environ (does not override existing env)
env_file = REPO / ".env"
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v
    print("Loaded .env:", env_file)
else:
    print("No .env at", env_file)

if not TOKEN:
    if "openrouter.ai" in BASE_URL:
        TOKEN = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY") or ""
    else:
        TOKEN = os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENROUTER_API_KEY") or ""
if not TOKEN and "localhost" not in BASE_URL and "127.0.0.1" not in BASE_URL and "ngrok" not in BASE_URL:
    raise RuntimeError("TOKEN trống — điền doan/.env (OPENROUTER_API_KEY=...) hoặc gán TOKEN")
if not TOKEN:
    TOKEN = "sk-local"  # self-host / ngrok dummy

print("MODEL:", MODEL)
print("BASE_URL:", BASE_URL)
print("MODE:", MODE)
print("N_SAMPLES:", N_SAMPLES)
print("MAX_CONCURRENCY:", MAX_CONCURRENCY)
print("REQUEST_DELAY_SEC:", REQUEST_DELAY_SEC)
print("TEMPERATURE:", TEMPERATURE, "(None → auto open/closed)")
print("REASONING_EFFORT:", REASONING_EFFORT)
print("TOKEN set:", bool(TOKEN), "(len=", len(TOKEN), ")")


REPO: /Users/nguyenkz/Documents/code/CS2202.CH202/doan
Loaded .env: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/.env
MODEL: google/gemini-3.6-flash
BASE_URL: https://openrouter.ai/api/v1
MODE: T
N_SAMPLES: 10
MAX_CONCURRENCY: 1
REQUEST_DELAY_SEC: 0.5
TEMPERATURE: 1.5 (None → auto open/closed)
REASONING_EFFORT: medium
TOKEN set: True (len= 73 )


In [4]:
# === Fetch OpenRouter providers for MODEL (price + latency) ===
# API: GET /api/v1/models/{model}/endpoints — không tốn token LLM.

import json
import urllib.error
import urllib.request
import pandas as pd
from IPython.display import display

PROVIDER_DF = pd.DataFrame()  # empty if not OpenRouter

if "openrouter.ai" not in (BASE_URL or ""):
    print("Skip provider fetch — BASE_URL không phải OpenRouter.")
else:
    endpoints_url = f"https://openrouter.ai/api/v1/models/{MODEL}/endpoints"
    req = urllib.request.Request(
        endpoints_url,
        headers={
            "Authorization": f"Bearer {TOKEN}",
            "Accept": "application/json",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenRouter endpoints HTTP {e.code}: {body}") from e

    endpoints = (payload.get("data") or {}).get("endpoints") or []
    rows = []
    for ep in endpoints:
        pricing = ep.get("pricing") or {}
        lat = ep.get("latency_last_30m") or {}
        thr = ep.get("throughput_last_30m") or {}
        prompt = float(pricing.get("prompt") or 0)
        completion = float(pricing.get("completion") or 0)
        rows.append({
            "provider": ep.get("provider_name"),
            "quant": ep.get("quantization"),
            "status": ep.get("status"),
            "prompt_$/1M": round(prompt * 1_000_000, 4),
            "completion_$/1M": round(completion * 1_000_000, 4),
            "latency_p50_ms": lat.get("p50"),
            "latency_p90_ms": lat.get("p90"),
            "throughput_p50": thr.get("p50"),
            "uptime_30m_%": None if ep.get("uptime_last_30m") is None else round(float(ep["uptime_last_30m"]), 2),
            "context": ep.get("context_length"),
            "tag": ep.get("tag"),
        })

    PROVIDER_DF = pd.DataFrame(rows)
    if len(PROVIDER_DF) == 0:
        print(f"No endpoints for MODEL={MODEL}")
    else:
        # cheapest + fastest first for browsing
        PROVIDER_DF = PROVIDER_DF.sort_values(
            by=["prompt_$/1M", "latency_p50_ms"],
            ascending=[True, True],
            na_position="last",
        ).reset_index(drop=True)
        print(f"MODEL={MODEL}  n_providers={len(PROVIDER_DF)}")
        display(PROVIDER_DF)
        print("Copy provider names vào cell config bên dưới (thứ tự = ưu tiên).")


MODEL=google/gemini-3.6-flash  n_providers=6


,provider,quant,status,prompt_$/1M,completion_$/1M,latency_p50_ms,latency_p90_ms,throughput_p50,uptime_30m_%,context,tag
0,Google AI Studio,unknown,0,0.75,3.75,2364.0,4009.9,85.5,97.51,1048576,google-ai-studio/flex
1,Google,unknown,0,0.75,3.75,7349.5,8396.7,10.0,99.88,1048576,google-vertex/global/flex
2,Google,unknown,0,1.50,7.50,1572.0,2871.7,108.0,99.88,1048576,google-vertex/global
3,Google AI Studio,unknown,0,1.50,7.50,1955.0,3113.3,156.0,97.51,1048576,google-ai-studio
4,Google,unknown,0,2.70,13.50,1578.0,3594.8,157.0,99.88,1048576,google-vertex/global/priority
5,Google AI Studio,unknown,0,2.70,13.50,NaN,NaN,NaN,97.51,1048576,google-ai-studio/priority


Copy provider names vào cell config bên dưới (thứ tự = ưu tiên).


In [5]:
# === Config OpenRouter providers (sau khi xem bảng ở cell trên) ===
# Thứ tự list = ưu tiên routing (provider.order). [] / None = để OpenRouter tự chọn.
# Tên phải khớp cột `provider` trong PROVIDER_DF (vd. "DeepInfra", "Baidu").
PROVIDERS = {
    "deepseek/deepseek-v4-flash": ["StreamLake", "DeepInfra", "Baidu"],
    "moonshotai/kimi-k3": ["Moonshot AI"],
    "z-ai/glm-5.2": ["StreamLake", "Novita", "Baidu"],
    "openai/gpt-4.1-mini": ["OpenAI"],
    "openai/gpt-5.6-sol": ["OpenAI"],
    "google/gemini-3.6-flash": ["google-ai-studio","google-vertex"],
}


OPENROUTER_PROVIDERS = PROVIDERS[MODEL]
OPENROUTER_ALLOW_FALLBACKS = True

# Gợi ý nhanh từ bảng vừa fetch (bỏ comment nếu muốn dùng):
# if len(PROVIDER_DF):
#     OPENROUTER_PROVIDERS = PROVIDER_DF["provider"].head(4).tolist()  # 4 rẻ/nhanh nhất

print("OPENROUTER_PROVIDERS:", OPENROUTER_PROVIDERS)
print("OPENROUTER_ALLOW_FALLBACKS:", OPENROUTER_ALLOW_FALLBACKS)
if len(PROVIDER_DF) and OPENROUTER_PROVIDERS:
    known = set(PROVIDER_DF["provider"].dropna().astype(str))
    unknown = [p for p in OPENROUTER_PROVIDERS if p not in known]
    if unknown:
        print("⚠ Không thấy trong endpoints (check spelling):", unknown)
    else:
        print("✓ Tất cả providers có trong endpoints của", MODEL)


OPENROUTER_PROVIDERS: ['google-ai-studio', 'google-vertex']
OPENROUTER_ALLOW_FALLBACKS: True
⚠ Không thấy trong endpoints (check spelling): ['google-ai-studio', 'google-vertex']


In [6]:
import sys
from pathlib import Path

assert REPO is not None, "Chạy cell setup trước (REPO)"
_src = Path(REPO) / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from plausibility_eval.run_eval import run_evaluation

result = run_evaluation(
    model=MODEL,
    token=TOKEN,
    base_url=BASE_URL,
    mode=MODE,
    repo=REPO,
    resume=True,
    n_samples_override=N_SAMPLES,
    max_concurrency=MAX_CONCURRENCY,
    request_delay_sec=REQUEST_DELAY_SEC,
    temperature_override=TEMPERATURE,
    reasoning_effort=REASONING_EFFORT,
    openrouter_providers=OPENROUTER_PROVIDERS,
    openrouter_allow_fallbacks=OPENROUTER_ALLOW_FALLBACKS,
)
print("out_dir:", result["out_dir"])
print("metrics:", result["metrics"])


[eval] model=google/gemini-3.6-flash mode=T provider=openrouter
       sentences=50 n_samples=10 → 500 API calls (full)
       resume=True skip_done=0 todo=50 remaining_calls=300
       temp=1.5 max_tokens=1024 concurrency=1 reasoning=medium
       request_delay_sec=0.5 rate_limit_retries=6
       providers=['google-ai-studio', 'google-vertex'] out=/Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/google__gemini-3.6-flash/T


/Users/nguyenkz/Documents/code/CS2202.CH202/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[eval]   → sample_id=s1_all mean=6.0 parse_fail=0 (saved)                                             
[eval]   → sample_id=s1_global mean=5.2 parse_fail=0 (saved)                                              
[eval]   → sample_id=s1_animate mean=6.6 parse_fail=0 (saved)                                              
[eval]   → sample_id=s1_plural mean=5.0 parse_fail=0 (saved)                                               
[eval]   → sample_id=s1_name mean=5.2 parse_fail=0 (saved)                                                
[eval]   → sample_id=s2_all mean=6.6 parse_fail=0 (saved)                                               
[eval]   → sample_id=s2_global mean=6.3 parse_fail=0 (saved)                                              
[eval]   → sample_id=s2_animate mean=3.0 parse_fail=0 (saved)                                              
[eval]   → sample_id=s2_plural mean=6.0 parse_fail=0 (saved)                                               
[eval]   → sample_id=s2_name mean=6.1 p

In [7]:
# Quick evidence check
from pathlib import Path
import json
out = Path(result["out_dir"])
calls = list[Path]((out / "calls").glob("*.json"))
print("n call files:", len(calls))
if calls:
    sample = json.loads(calls[0].read_text())
    print("keys:", sorted(sample.keys()))
    print("trace_id:", sample.get("trace_id"))
    print("usage:", sample.get("usage"))
    print("has request:", bool(sample.get("request")))
    print("has output_text:", bool(sample.get("output_text")))
    print("reasoning_text is None?:", sample.get("reasoning_text") is None)
print("scores.jsonl lines:", sum(1 for _ in open(out / "scores.jsonl")))
assert "cost_usd" not in (out / "scores.jsonl").read_text()[:2000]
print("OK: no cost_usd in scores head")


n call files: 500
keys: ['call_index', 'condition_id', 'created_at', 'latency_ms', 'model_id', 'output_text', 'parse_ok', 'parsed_score', 'provider', 'reasoning_text', 'request', 'request_id', 'response_raw', 'sample_id', 'trace_id', 'usage']
trace_id: gen-1785073114-VZHLRO9Yhs9wBpEeMMwo
usage: {'input_tokens': 704, 'output_tokens': 442, 'reasoning_tokens': 410}
has request: True
has output_text: True
reasoning_text is None?: False
scores.jsonl lines: 50
OK: no cost_usd in scores head


## Full matrix

Đổi `MODE` (và/hoặc `MODEL` / `BASE_URL`) → chạy lại.  
Thứ tự gợi ý trên model chính: ORIG → S → T → ST → ST-E (`configs/model_coverage.yaml`).
